# PuoBERTa + Masked Rationale Prediction (MRP) for Setswana Offensive Language Detection

This notebook extends the published `setswana-offensive-977` companion repository workflow.

**What is inherited from the companion workflow**

- PuoBERTa backbone: `dsfsi/PuoBERTa`
- Maximum length: `128`
- Learning rate: `1e-5`
- Weight decay: `0.01`
- CV epochs: `5`
- Final training epochs: `10`
- Training batch size: `16`
- Evaluation batch size: `64`
- Class weights: `[1.0, 2.0]`
- Seeds: `[42, 0, 13]`
- 80/20 split with an untouched tag-free holdout
- Five-fold stratified CV on the 80% development partition

**What is new in this notebook**

Semantic trigger spans are no longer used as visible tags for the classifier. Instead, they are converted into **gold rationale supervision** for an auxiliary **Masked Rationale Prediction (MRP)** objective.

The model receives:

1. a **tag-free classification view** of each sentence for offensive/non-offensive prediction; and
2. a **rationale-masked MRP view** in which the gold rationale token positions are replaced by `<mask>` and predicted as an auxiliary task.

The final holdout evaluation remains strictly tag-free.

In [ ]:
# Optional: uncomment this cell in a fresh Colab runtime.
!pip -q install transformers datasets accelerate evaluate scikit-learn pandas numpy scipy lime pyyaml

## 1. Imports and global configuration

In [ ]:
import os
import re
import json
import math
import random
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

from transformers import (
    RobertaTokenizerFast,
    AutoModelForMaskedLM,
    TrainingArguments,
    Trainer,
    default_data_collator,
    set_seed,
)
from transformers.modeling_outputs import SequenceClassifierOutput

warnings.filterwarnings("ignore")

In [ ]:
import os
import re
import json
import math
import random
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

from transformers import (
    RobertaTokenizerFast,
    AutoModelForMaskedLM,
    TrainingArguments,
    Trainer,
    default_data_collator,
    set_seed,
)
from transformers.modeling_outputs import SequenceClassifierOutput

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------
# Inherited companion-repository configuration
# ---------------------------------------------------------------------
MODEL_NAME = "dsfsi/PuoBERTa"
MAX_LENGTH = 128
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
EPOCHS_CV = 5
EPOCHS_FINAL = 10
BATCH_TRAIN = 16
BATCH_EVAL = 64
CLASS_WEIGHTS = [1.0, 2.0]
CV_SEEDS = [42, 0, 13]
SLIME_SEEDS = [42, 0, 13, 77, 99]

# New MRP-specific setting.
# Keep this fixed initially for a clean extension experiment. Tune only in a later ablation.
LAMBDA_MRP = 0.5

# Evaluation protocol
# NOTE: the repository now provides a separate development/training file with triggers
# and a separate tag-free test file. Therefore, we no longer create an 80/20 split here.
N_SPLITS = 5
BOOTSTRAP_RESAMPLES = 1000

# Paths
# REPO_ROOT = Path("..").resolve() # This was commented out in a previous diff, but Path is still imported.
OUTPUT_DIR = Path("/content/drive/My Drive/Colab Notebooks/mrp_puoberta") # Converted to Path object
#OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# Dataset loading options
# ---------------------------------------------------------------------
# The expected release format for this MRP notebook is:
#   1) TRAIN_DATA_PATH: training/development set WITH <TRIGGER>...</TRIGGER> markup
#      or with equivalent trigger/rationale metadata.
#   2) TEST_DATA_PATH: final holdout/test set WITHOUT visible trigger markup.
#
# Configure either the local file paths below or a Hugging Face DatasetDict with
# explicit train/test split names.
TRAIN_DATA_PATH = "/content/drive/My Drive/Colab Notebooks/training_data.csv"
TEST_DATA_PATH = "/content/drive/My Drive/Colab Notebooks/TEST_WITHOUT_TRIGGERS.csv"
TEST_WITH_TRIGGERS_PATH = "/content/drive/My Drive/Colab Notebooks/TEST_WITH_TRIGGERS.csv"


HF_DATASET_ID = None    # e.g., "bkekgathetse/setswana-offensive-977"
HF_TRAIN_SPLIT = "train"
HF_TEST_SPLIT = "test"

# If your dataset columns use known names, set them here. Otherwise the notebook will infer likely columns.
TEXT_COLUMN = None
LABEL_COLUMN = None
TRIGGER_COLUMN = None

## 2. Utility functions

In [ ]:
def seed_everything(seed: int) -> None:
    """Set Python, NumPy, PyTorch, and Transformers seeds."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)


def softmax_np(logits: np.ndarray) -> np.ndarray:
    logits = np.asarray(logits)
    z = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(z)
    return exp / exp.sum(axis=1, keepdims=True)


def infer_column(df: pd.DataFrame, candidates: List[str], label: str) -> str:
    """Infer a likely column name from a list of candidates."""
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    raise ValueError(
        f"Could not infer {label} column. Available columns: {list(df.columns)}. "
        f"Set {label.upper()}_COLUMN manually."
    )


def normalize_label(value: Any) -> int:
    """Normalize common offensive/non-offensive labels to 0/1."""
    if pd.isna(value):
        raise ValueError("Missing label encountered.")
    if isinstance(value, (int, np.integer)):
        return int(value)
    if isinstance(value, float) and value.is_integer():
        return int(value)
    s = str(value).strip().lower()
    if s in {"1", "offensive", "off", "abusive", "hate", "toxic", "yes", "true"}:
        return 1
    if s in {"0", "non-offensive", "non offensive", "not offensive", "none", "neutral", "no", "false"}:
        return 0
    raise ValueError(f"Unsupported label value: {value!r}")

## 3. Load source splits

The current repository workflow uses two separate data artefacts:

- a **training/development set with trigger annotations**. This file is used to extract gold rationale spans for MRP supervision;
- a **tag-free test/holdout set**. This file is used only for final classification evaluation and must not contain visible `<TRIGGER>...</TRIGGER>` markup.

This design preserves the anti-leakage principle from the companion study: trigger annotations are used as supervision during training, but they are not exposed to the classifier at final test time.

Expected minimum fields:

- a text column;
- a binary label column;
- for the training/development set only, either inline `<TRIGGER>...</TRIGGER>` markup or a separate trigger/rationale column.


In [ ]:
def read_table(path_like: Any) -> pd.DataFrame:
    """Read a CSV/TSV/XLSX/JSON/JSONL file into a DataFrame."""
    path = Path(path_like)
    if not path.exists():
        raise FileNotFoundError(f"Dataset file does not exist: {path}")

    suffix = path.suffix.lower()
    if suffix in {".csv", ".tsv"}:
        sep = "\t" if suffix == ".tsv" else ","
        return pd.read_csv(path, sep=sep)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix in {".json", ".jsonl"}:
        return pd.read_json(path, lines=suffix == ".jsonl")

    raise ValueError(f"Unsupported file type: {path.suffix}")


def contains_trigger_markup(df: pd.DataFrame, text_col: Optional[str] = None) -> bool:
    """Check whether a dataframe text column contains visible trigger tags."""
    if df.empty:
        return False

    col = text_col
    if col is None:
        try:
            col = TEXT_COLUMN or infer_column(
                df,
                ["text", "TEXT", "sentence", "Sentence", "content", "clean_text", "text_with_triggers", "Text"],
                "text",
            )
        except ValueError:
            return False

    return df[col].astype(str).str.contains(r"</?TRIGGER>", case=False, regex=True).any()


def load_source_dataframes() -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Load the two required source splits.

    Returns
    -------
    raw_train_df : pd.DataFrame
        Training/development set containing trigger annotations for rationale supervision.
    raw_test_df : pd.DataFrame
        Tag-free holdout/test set used for final evaluation.
    """
    # Option A: local files.
    if TRAIN_DATA_PATH is not None or TEST_DATA_PATH is not None:
        if TRAIN_DATA_PATH is None or TEST_DATA_PATH is None:
            raise ValueError(
                "Both TRAIN_DATA_PATH and TEST_DATA_PATH must be set. "
                "The MRP workflow expects a trigger-annotated training/development file "
                "and a separate tag-free test file."
            )

        raw_train_df = read_table(TRAIN_DATA_PATH)
        raw_test_df = read_table(TEST_DATA_PATH)

    # Option B: Hugging Face DatasetDict with explicit train/test splits.
    elif HF_DATASET_ID is not None:
        try:
            from datasets import load_dataset
        except ImportError as exc:
            raise ImportError("Install `datasets` or use TRAIN_DATA_PATH and TEST_DATA_PATH instead.") from exc

        ds = load_dataset(HF_DATASET_ID)
        available = list(ds.keys())
        if HF_TRAIN_SPLIT not in ds or HF_TEST_SPLIT not in ds:
            raise ValueError(
                f"Expected splits '{HF_TRAIN_SPLIT}' and '{HF_TEST_SPLIT}' in {HF_DATASET_ID}, "
                f"but found: {available}. Set HF_TRAIN_SPLIT and HF_TEST_SPLIT correctly."
            )

        raw_train_df = ds[HF_TRAIN_SPLIT].to_pandas()
        raw_test_df = ds[HF_TEST_SPLIT].to_pandas()

    else:
        raise ValueError(
            "No dataset source configured. Set TRAIN_DATA_PATH and TEST_DATA_PATH for local files, "
            "or set HF_DATASET_ID with HF_TRAIN_SPLIT and HF_TEST_SPLIT."
        )

    # Sanity warnings: training/development data should contain rationale evidence;
    # final holdout/test data should be tag-free.
    if not contains_trigger_markup(raw_train_df) and TRIGGER_COLUMN is None:
        warnings.warn(
            "The training/development dataframe does not appear to contain inline <TRIGGER> tags, "
            "and TRIGGER_COLUMN is not set. MRP rationale spans may be empty unless a trigger column "
            "is inferred automatically in prepare_dataframe()."
        )

    if contains_trigger_markup(raw_test_df):
        raise ValueError(
            "The test/holdout dataframe contains visible <TRIGGER> markup. "
            "Use the tag-free test file for final evaluation."
        )

    return raw_train_df, raw_test_df


# Uncomment after configuring TRAIN_DATA_PATH and TEST_DATA_PATH, or HF_DATASET_ID.ASET_ID.

raw_train_df, raw_test_df = load_source_dataframes()
print("Raw train/dev shape:", raw_train_df.shape)
print("Raw tag-free test shape:", raw_test_df.shape)
display(raw_train_df.head())
display(raw_test_df.head())


## 4. Trigger extraction and rationale span preparation

The MRP paper must not train the classifier on visible trigger markup. This section extracts the trigger spans, removes the tags, and keeps the rationale boundaries for the auxiliary MRP task.

In [ ]:
TRIGGER_PATTERN = re.compile(r"<TRIGGER>(.*?)</TRIGGER>", flags=re.IGNORECASE | re.DOTALL)


def extract_inline_trigger_spans(text: str) -> Tuple[str, List[Tuple[int, int]]]:
    """
    Convert text with inline <TRIGGER>...</TRIGGER> markup into:
      - clean text with tags removed;
      - character spans of trigger content in the clean text.

    Multiple trigger spans are supported.
    """
    text = str(text)
    spans: List[Tuple[int, int]] = []
    clean_parts: List[str] = []
    cursor = 0
    clean_cursor = 0

    for match in TRIGGER_PATTERN.finditer(text):
        before = text[cursor:match.start()]
        clean_parts.append(before)
        clean_cursor += len(before)

        trigger_text = match.group(1)
        start = clean_cursor
        clean_parts.append(trigger_text)
        clean_cursor += len(trigger_text)
        end = clean_cursor
        spans.append((start, end))

        cursor = match.end()

    tail = text[cursor:]
    clean_parts.append(tail)
    clean_text = "".join(clean_parts)
    return clean_text, spans


def find_trigger_text_spans(clean_text: str, trigger_value: Any) -> List[Tuple[int, int]]:
    """
    Locate trigger/rationale text inside a clean sentence when triggers are stored in a separate column.

    Supports simple delimiters: ||, ;, comma, or JSON-style list strings.
    """
    if trigger_value is None or pd.isna(trigger_value):
        return []

    raw = str(trigger_value).strip()
    if not raw:
        return []

    triggers: List[str] = []
    try:
        parsed = json.loads(raw)
        if isinstance(parsed, list):
            triggers = [str(x) for x in parsed]
        else:
            triggers = [str(parsed)]
    except Exception:
        if "||" in raw:
            triggers = [x.strip() for x in raw.split("||")]
        elif ";" in raw:
            triggers = [x.strip() for x in raw.split(";")]
        elif "," in raw and len(raw.split(",")) <= 5:
            triggers = [x.strip() for x in raw.split(",")]
        else:
            triggers = [raw]

    spans: List[Tuple[int, int]] = []
    lowered = clean_text.lower()
    used_until = 0
    for trig in triggers:
        trig = trig.strip()
        if not trig:
            continue
        idx = lowered.find(trig.lower(), used_until)
        if idx == -1:
            idx = lowered.find(trig.lower())
        if idx != -1:
            spans.append((idx, idx + len(trig)))
            used_until = idx + len(trig)
    return spans


def prepare_dataframe(
    df: pd.DataFrame,
    split_name: str,
    require_rationales: bool = False,
) -> pd.DataFrame:
    """
    Create clean_text, label, and char_spans columns.

    Parameters
    ----------
    df : pd.DataFrame
        Raw dataframe for one source split.
    split_name : str
        Human-readable split name, e.g. "train_dev" or "holdout".
    require_rationales : bool
        If True, warn when offensive rows do not have rationale spans. Use True for
        the trigger-annotated training/development file and False for the tag-free test file.
    """
    df = df.copy()

    text_col = TEXT_COLUMN or infer_column(
        df,
        ["text", "TEXT", "sentence", "Sentence", "content", "clean_text", "text_with_triggers", "Text"],
        "text",
    )
    label_col = LABEL_COLUMN or infer_column(
        df,
        ["label", "LABEL", "target", "class", "category", "offensive", "Label"],
        "label",
    )

    trigger_col = TRIGGER_COLUMN
    if trigger_col is None:
        possible_trigger_cols = [
            "trigger", "triggers", "TRIGGER", "semantic_trigger", "semantic_triggers",
            "rationale", "rationales", "trigger_span", "trigger_spans", "span", "spans"
        ]
        trigger_col = next((c for c in df.columns if c.lower() in {x.lower() for x in possible_trigger_cols}), None)

    clean_texts, char_spans_all = [], []
    for _, row in df.iterrows():
        original_text = str(row[text_col])
        clean_text, inline_spans = extract_inline_trigger_spans(original_text)

        spans = inline_spans
        if not spans and trigger_col is not None:
            spans = find_trigger_text_spans(clean_text, row[trigger_col])

        clean_texts.append(clean_text)
        char_spans_all.append(spans)

    out = pd.DataFrame({
        "clean_text": clean_texts,
        "label": [normalize_label(x) for x in df[label_col]],
        "char_spans": char_spans_all,
        "source_split": split_name,
    })

    out["clean_text"] = out["clean_text"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    out = out[out["clean_text"].str.len() > 0].reset_index(drop=True)

    if require_rationales:
        missing = out[(out["label"] == 1) & (out["char_spans"].map(len) == 0)]
        if len(missing) > 0:
            warnings.warn(
                f"{len(missing)} offensive rows in split '{split_name}' have no detected rationale spans. "
                "Check trigger markup, TRIGGER_COLUMN, and span extraction before training."
            )

    return out


# Example after loading:
raw_train_df, raw_test_df = load_source_dataframes()
train_dev_df = prepare_dataframe(raw_train_df, split_name="train_dev", require_rationales=True)
holdout_df = prepare_dataframe(raw_test_df, split_name="holdout", require_rationales=False)
display(train_dev_df[["clean_text", "label", "char_spans", "source_split"]].head())
display(holdout_df[["clean_text", "label", "char_spans", "source_split"]].head())


## 5. Token-level rationale mapping and MRP dataset

This converts character-level rationale spans into tokenizer-level rationale positions. If a rationale word is split into multiple BPE/subword tokens, all corresponding subword tokens are treated as rationale tokens.

In [ ]:
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME)
MASK_TOKEN_ID = tokenizer.mask_token_id

assert tokenizer.is_fast, "MRP requires a fast tokenizer for offset mappings."
assert MASK_TOKEN_ID is not None, "Tokenizer must define a mask token for MRP."

print("Tokenizer class:", tokenizer.__class__.__name__)
print("Fast tokenizer:", tokenizer.is_fast)
print("Mask token:", tokenizer.mask_token, MASK_TOKEN_ID)
print("Pad token:", tokenizer.pad_token, tokenizer.pad_token_id)


def char_spans_to_token_indices(offset_mapping: List[Tuple[int, int]], char_spans: List[Tuple[int, int]]) -> List[int]:
    """Map character spans to token indices using tokenizer offset mappings."""
    rationale_indices = []
    for idx, (tok_start, tok_end) in enumerate(offset_mapping):
        if tok_start == tok_end:
            continue
        for span_start, span_end in char_spans:
            if tok_start < span_end and tok_end > span_start:
                rationale_indices.append(idx)
                break
    return sorted(set(rationale_indices))


class SetswanaMRPDataset(Dataset):
    """
    Dataset returning both:
      - classification inputs: input_ids_cls, attention_mask_cls
      - MRP inputs: input_ids_mrp, attention_mask_mrp, mlm_labels
      - classification labels: labels
    """
    def __init__(self, frame: pd.DataFrame, tokenizer: RobertaTokenizerFast, max_length: int = 128):
        self.frame = frame.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        row = self.frame.iloc[idx]
        text = str(row["clean_text"])
        label = int(row["label"])
        char_spans = row["char_spans"] if isinstance(row["char_spans"], list) else []

        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_offsets_mapping=True,
        )

        input_ids = torch.tensor(enc["input_ids"], dtype=torch.long)
        attention_mask = torch.tensor(enc["attention_mask"], dtype=torch.long)
        offset_mapping = enc["offset_mapping"]

        rationale_indices = char_spans_to_token_indices(offset_mapping, char_spans)

        input_ids_cls = input_ids.clone()
        attention_mask_cls = attention_mask.clone()

        input_ids_mrp = input_ids.clone()
        mlm_labels = torch.full_like(input_ids, fill_value=-100)

        if label == 1:
            for pos in rationale_indices:
                if attention_mask[pos].item() == 1 and input_ids[pos].item() not in self.tokenizer.all_special_ids:
                    mlm_labels[pos] = input_ids[pos]
                    input_ids_mrp[pos] = MASK_TOKEN_ID

        return {
            "input_ids_cls": input_ids_cls,
            "attention_mask_cls": attention_mask_cls,
            "input_ids_mrp": input_ids_mrp,
            "attention_mask_mrp": attention_mask.clone(),
            "mlm_labels": mlm_labels,
            "labels": torch.tensor(label, dtype=torch.long),
        }


## 6. Joint PuoBERTa + MRP model

The model uses one PuoBERTa masked-language-model backbone and adds a binary classification head. During training, the loss is:

\[
\mathcal{L} = \mathcal{L}_{cls} + \lambda \mathcal{L}_{mrp}.
\]

The classification branch uses the tag-free input. The MRP branch uses the rationale-masked input.

In [ ]:
class PuoBERTaForJointMRP(nn.Module):
    """PuoBERTa model with a binary classification head and auxiliary MRP loss.

    Important implementation detail:
    - The model keeps only ONE registered pretrained model: self.mlm_model.
    - We do NOT assign self.encoder = self.mlm_model.roberta, because that
      registers the same underlying tensors twice under different names.
    - During forward(), we access the RoBERTa encoder through a local variable.
      This avoids the shared-tensor checkpoint error during Trainer saving.
    """

    def __init__(
        self,
        model_name: str,
        num_labels: int = 2,
        class_weights: Optional[List[float]] = None,
        lambda_mrp: float = 0.5,
    ):
        super().__init__()
        self.mlm_model = AutoModelForMaskedLM.from_pretrained(model_name)
        self.config = self.mlm_model.config
        self.num_labels = num_labels
        self.lambda_mrp = lambda_mrp

        hidden_size = self.config.hidden_size
        dropout_prob = getattr(self.config, "hidden_dropout_prob", 0.1)
        self.dropout = nn.Dropout(dropout_prob)
        self.classifier = nn.Linear(hidden_size, num_labels)

        weights = torch.tensor(
            class_weights if class_weights is not None else [1.0] * num_labels,
            dtype=torch.float,
        )
        self.register_buffer("class_weights", weights)

    def _get_encoder(self):
        """Return the underlying RoBERTa encoder without registering a duplicate module."""
        base_prefix = getattr(self.mlm_model, "base_model_prefix", None)
        if base_prefix is not None and hasattr(self.mlm_model, base_prefix):
            return getattr(self.mlm_model, base_prefix)
        return self.mlm_model.base_model

    def forward(
        self,
        input_ids_cls=None,
        attention_mask_cls=None,
        input_ids_mrp=None,
        attention_mask_mrp=None,
        mlm_labels=None,
        labels=None,
        **kwargs,
    ):
        encoder = self._get_encoder()
        enc_outputs = encoder(
            input_ids=input_ids_cls,
            attention_mask=attention_mask_cls,
            return_dict=True,
        )
        cls_rep = enc_outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(self.dropout(cls_rep))

        loss = None
        if labels is not None:
            loss_cls = F.cross_entropy(
                logits,
                labels,
                weight=self.class_weights.to(logits.device),
            )

            if mlm_labels is not None and (mlm_labels != -100).any():
                mlm_outputs = self.mlm_model(
                    input_ids=input_ids_mrp,
                    attention_mask=attention_mask_mrp,
                    labels=mlm_labels,
                    return_dict=True,
                )
                loss_mrp = mlm_outputs.loss
            else:
                loss_mrp = torch.tensor(0.0, device=logits.device)

            loss = loss_cls + self.lambda_mrp * loss_mrp

        return SequenceClassifierOutput(loss=loss, logits=logits)


## 7. Metrics and bootstrap confidence intervals

In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
)

def compute_metrics(eval_pred):
    """
    Compute classification metrics from Hugging Face Trainer outputs.

    This version is robust to custom models that may return:
    - cls_logits only, or
    - a tuple such as (cls_logits, mrp_logits).
    """

    predictions, labels = eval_pred

    # If the model returns multiple outputs, take classification logits only.
    if isinstance(predictions, tuple):
        cls_logits = predictions[0]
    else:
        cls_logits = predictions

    # Handle cases where labels might be a tuple, e.g., (unique_labels, actual_labels)
    # The Trainer's EvalPrediction.label_ids is expected to be a np.ndarray.
    # If it's a tuple, it usually means (unique_classes_array, actual_labels_array).
    if isinstance(labels, tuple):
        # We want the actual ground truth labels, which are typically the second element
        # if the tuple is (unique_classes, actual_labels)
        if len(labels) > 1 and isinstance(labels[-1], np.ndarray):
            labels = labels[-1]
        elif len(labels) == 1: # Handle case where it's (actual_labels_array,)
            labels = labels[0]
        # If it's still a tuple of unexpected structure, np.asarray will try its best
        # or fail as before, which is fine as a fallback.

    cls_logits = np.asarray(cls_logits)
    labels = np.asarray(labels)

    # Expected cls_logits shape: (num_examples, 2)
    # Correct argmax is across the class dimension.
    preds = np.argmax(cls_logits, axis=-1)

    # Debug check: remove after confirming everything works.
    if len(preds) != len(labels):
        raise ValueError(
            f"Prediction/label length mismatch. "
            f"cls_logits shape={cls_logits.shape}, "
            f"preds shape={preds.shape}, "
            f"labels shape={labels.shape}. "
            f"Check argmax axis and model outputs."
        )

    accuracy = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        labels=[0, 1],
        average=None,
        zero_division=0,
    )

    mcc = matthews_corrcoef(labels, preds)

    # ROC-AUC requires probabilities for the offensive class.
    try:
        exp_logits = np.exp(cls_logits - np.max(cls_logits, axis=1, keepdims=True))
        probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)
        offensive_probs = probs[:, 1]
        roc_auc = roc_auc_score(labels, offensive_probs)
    except Exception:
        roc_auc = np.nan

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "precision_0": precision[0],
        "recall_0": recall[0],
        "f1_0": f1[0],
        "precision_1": precision[1],
        "recall_1": recall[1],
        "f1_1": f1[1],
        "mcc": mcc,
        "roc_auc": roc_auc,
    }

## 8. Train/evaluate helpers

In [ ]:
def _make_training_arguments_with_strategy(strategy_value: str, **kwargs) -> TrainingArguments:
    """Create TrainingArguments safely across different Transformers versions.

    This helper handles three version differences that commonly break notebooks:
    1. Some versions use `evaluation_strategy`; newer versions may expose `eval_strategy`.
    2. Older versions do not support `save_safetensors`.
    3. Any unsupported keyword is removed before constructing TrainingArguments.
    """
    import inspect

    params = inspect.signature(TrainingArguments.__init__).parameters

    # Transformers versions differ on this keyword name.
    strategy_key = "eval_strategy" if "eval_strategy" in params else "evaluation_strategy"
    kwargs[strategy_key] = strategy_value

    # Drop unsupported kwargs for the installed Transformers version.
    unsupported = sorted(k for k in list(kwargs.keys()) if k not in params)
    for k in unsupported:
        kwargs.pop(k, None)

    if unsupported:
        print(
            "TrainingArguments: removed unsupported argument(s) for this "
            f"Transformers version: {unsupported}"
        )

    return TrainingArguments(**kwargs)


def make_training_args(
    output_dir: Path,
    epochs: int,
    seed: int,
    eval_during_training: bool = True,
) -> TrainingArguments:
    """Create TrainingArguments compatible with the companion workflow.

    Cross-validation evaluates each epoch and, where supported, loads the best
    fold model according to offensive-class recall. Final training does not use
    the separate tag-free holdout for checkpoint selection.

    The argument `save_safetensors=False` is added only if the installed
    Transformers version supports it. This avoids a TypeError on older Colab
    runtimes while still preventing shared-tensor safetensors errors on newer
    runtimes.
    """
    import inspect

    params = inspect.signature(TrainingArguments.__init__).parameters

    common_kwargs = dict(
        output_dir=str(output_dir),
        num_train_epochs=epochs,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        per_device_train_batch_size=BATCH_TRAIN,
        per_device_eval_batch_size=BATCH_EVAL,
        logging_strategy="epoch",
        seed=seed,
        report_to="none",
        remove_unused_columns=False,
        fp16=torch.cuda.is_available(),
        save_total_limit=1,
    )

    # Newer Transformers versions support this argument; older ones do not.
    # When available, setting it to False avoids shared-tensor checkpoint errors
    # for models with tied/shared MLM embeddings.
    if "save_safetensors" in params:
        common_kwargs["save_safetensors"] = False

    if eval_during_training:
        return _make_training_arguments_with_strategy(
            "epoch",
            **common_kwargs,
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="recall_1",
            greater_is_better=True,
        )

    # Final training must not use the holdout set for evaluation/model selection.
    # We also avoid unnecessary checkpointing here; the custom save_mrp_checkpoint()
    # function is used after training to save the final model safely.
    return _make_training_arguments_with_strategy(
        "no",
        **common_kwargs,
        save_strategy="no",
        load_best_model_at_end=False,
    )


def build_trainer(
    train_df: pd.DataFrame,
    eval_df: Optional[pd.DataFrame],
    output_dir: Path,
    epochs: int,
    seed: int,
    eval_during_training: bool = True,
) -> Trainer:
    """Build a Hugging Face Trainer for the joint PuoBERTa-MRP model."""
    seed_everything(seed)

    model = PuoBERTaForJointMRP(
        model_name=MODEL_NAME,
        num_labels=2,
        class_weights=CLASS_WEIGHTS,
        lambda_mrp=LAMBDA_MRP,
    )

    train_dataset = SetswanaMRPDataset(train_df, tokenizer, max_length=MAX_LENGTH)
    eval_dataset = None if eval_df is None else SetswanaMRPDataset(eval_df, tokenizer, max_length=MAX_LENGTH)

    args = make_training_args(
        output_dir=output_dir,
        epochs=epochs,
        seed=seed,
        eval_during_training=eval_during_training,
    )

    return Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=default_data_collator,
        compute_metrics=compute_metrics if eval_dataset is not None else None,
    )


def predict_logits(trainer: Trainer, frame: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Predict logits, probabilities, and labels for a prepared dataframe."""
    dataset = SetswanaMRPDataset(frame, tokenizer, max_length=MAX_LENGTH)
    pred = trainer.predict(dataset)
    logits = pred.predictions

    # If a future model variant returns a tuple, keep the classification logits only.
    if isinstance(logits, tuple):
        logits = logits[0]

    probs = softmax_np(logits)
    preds = probs.argmax(axis=1)
    return logits, probs, preds

## 9. Main data preparation from released train/test artefacts

Run this section after setting `TRAIN_DATA_PATH` and `TEST_DATA_PATH`, or after setting `HF_DATASET_ID` with the correct `HF_TRAIN_SPLIT` and `HF_TEST_SPLIT`.

The notebook expects:
- a training/development file **with trigger annotations** for MRP supervision;
- a separate final test/holdout file **without visible trigger markup**.

No new 80/20 split is created here because the repository already provides the train/test artefacts.

In [ ]:
# 1) Load the two released source splits.
#    - raw_train_df: training/development data WITH trigger annotations for MRP supervision.
#    - raw_test_df: final tag-free holdout/test data WITHOUT visible trigger markup.
raw_train_df, raw_test_df = load_source_dataframes()

# 2) Prepare each split separately.
#    Do not create a new 80/20 split here: the repository already provides separate train/test artefacts.
train_dev_df = prepare_dataframe(raw_train_df, split_name="train_dev", require_rationales=True)
holdout_df = prepare_dataframe(raw_test_df, split_name="holdout", require_rationales=False)

# 3) Sanity checks.
print("Development split:", train_dev_df.shape, train_dev_df.label.value_counts().to_dict())
print("Tag-free holdout split:", holdout_df.shape, holdout_df.label.value_counts().to_dict())
print("Offensive train/dev rows with rationale spans:",train_dev_df[(train_dev_df.label == 1) & (train_dev_df.char_spans.map(len) > 0)].shape[0],)
print("Holdout rows with rationale spans, expected 0 unless hidden rationale metadata is supplied:",holdout_df[holdout_df.char_spans.map(len) > 0].shape[0],
 )
display(train_dev_df.head())
display(holdout_df.head())


In [ ]:
import json
import torch
from pathlib import Path

def save_mrp_checkpoint(trainer, tokenizer, output_dir, extra_config=None):
    """
    Save the custom PuoBERTa-MRP model safely.

    This saves:
    - model state_dict
    - tokenizer
    - training/MRP configuration
    - Trainer state
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Save model weights for the custom MRP architecture
    torch.save(
        trainer.model.state_dict(),
        output_dir / "pytorch_model.bin"
    )

    # Save tokenizer files
    tokenizer.save_pretrained(output_dir)

    # Save model/training configuration
    config_to_save = {
        "model_name": MODEL_NAME,
        "architecture": "PuoBERTaMRPForSequenceClassification",
        "num_labels": 2,
        "max_length": MAX_LENGTH,
        "lambda_mrp": LAMBDA_MRP,
        "class_weights": CLASS_WEIGHTS,
        "label_mapping": {
            "0": "non-offensive",
            "1": "offensive"
        },
        "note": (
            "Custom MRP model. Reload by instantiating the same model class "
            "and loading pytorch_model.bin with load_state_dict."
        )
    }

    if extra_config is not None:
        config_to_save.update(extra_config)

    with open(output_dir / "mrp_config.json", "w", encoding="utf-8") as f:
        json.dump(config_to_save, f, indent=2, ensure_ascii=False)

    # Save Trainer state if available
    try:
        trainer.save_state()
    except Exception as exc:
        print(f"Trainer state was not saved: {exc}")

    print(f"Saved custom MRP checkpoint to: {output_dir}")

## 10. Five-fold stratified CV on the 80% development split

This mirrors the companion protocol but trains the MRP-augmented classifier.

In [ ]:
import inspect
from transformers import TrainingArguments, Trainer # Added Trainer here for local build_trainer_local

def run_cross_validation(train_dev_df: pd.DataFrame) -> pd.DataFrame:
    """
    Run stratified 5-fold cross-validation on the trigger-annotated
    training/development set.

    This function is deliberately defensive:
    - ensures train_dev_df is a pandas DataFrame;
    - verifies that required columns exist;
    - uses row indices for StratifiedKFold instead of passing the dataframe
      directly as X;
    - checks that each class has enough samples for N_SPLITS.
    """

    # --- Start of local redefinitions for error fix ---
    def _make_training_arguments_with_strategy_local(strategy_value: str, **kwargs) -> TrainingArguments:
        params = inspect.signature(TrainingArguments.__init__).parameters
        strategy_key = "eval_strategy" if "eval_strategy" in params else "evaluation_strategy"
        kwargs[strategy_key] = strategy_value
        return TrainingArguments(**kwargs)

    def make_training_args_local(
        output_dir: Path,
        epochs: int,
        seed: int,
        eval_during_training: bool = True,
    ) -> TrainingArguments:
        common_kwargs = dict(
            output_dir=str(output_dir),
            num_train_epochs=epochs,
            learning_rate=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
            per_device_train_batch_size=BATCH_TRAIN,
            per_device_eval_batch_size=BATCH_EVAL,
            logging_strategy="epoch",
            seed=seed,
            report_to="none",
            remove_unused_columns=False,
            fp16=torch.cuda.is_available(),
            save_total_limit=1,
        )

        # Conditionally add save_safetensors only if supported by the TrainingArguments constructor
        if "save_safetensors" in inspect.signature(TrainingArguments.__init__).parameters:
            common_kwargs["save_safetensors"] = False

        if eval_during_training:
            return _make_training_arguments_with_strategy_local(
                "epoch",
                **common_kwargs,
                save_strategy="epoch",
                load_best_model_at_end=True,
                metric_for_best_model="recall_1",
                greater_is_better=True,
            )

        return _make_training_arguments_with_strategy_local(
            "no",
            **common_kwargs,
            save_strategy="epoch",
            load_best_model_at_end=False,
        )

    # Redefine build_trainer locally to use the fixed make_training_args_local
    def build_trainer_local(
        train_df: pd.DataFrame,
        eval_df: Optional[pd.DataFrame],
        output_dir: Path,
        epochs: int,
        seed: int,
        eval_during_training: bool = True,
    ) -> Trainer:
        # Assuming seed_everything, PuoBERTaForJointMRP, SetswanaMRPDataset,
        # tokenizer, MAX_LENGTH, CLASS_WEIGHTS, LAMBDA_MRP, default_data_collator,
        # compute_metrics are all available from the outer scope as they were before.
        seed_everything(seed)

        model = PuoBERTaForJointMRP(
            model_name=MODEL_NAME,
            num_labels=2,
            class_weights=CLASS_WEIGHTS,
            lambda_mrp=LAMBDA_MRP,
        )

        train_dataset = SetswanaMRPDataset(train_df, tokenizer, max_length=MAX_LENGTH)
        eval_dataset = None if eval_df is None else SetswanaMRPDataset(eval_df, tokenizer, max_length=MAX_LENGTH)

        args = make_training_args_local( # Use the locally defined fixed version
            output_dir=output_dir,
            epochs=epochs,
            seed=seed,
            eval_during_training=eval_during_training,
        )

        trainer_kwargs = {
            "model": model,
            "args": args,
            "train_dataset": train_dataset,
            "eval_dataset": eval_dataset,
            "data_collator": default_data_collator,
            "compute_metrics": compute_metrics if eval_dataset is not None else None,
        }

        # Conditionally add use_safetensors only if supported by the Trainer constructor
        if "use_safetensors" in inspect.signature(Trainer.__init__).parameters:
            trainer_kwargs["use_safetensors"] = False

        trainer = Trainer(**trainer_kwargs)

        # Override the save_model method to force torch.save
        # This is a workaround for transformers versions that may still try to use
        # safetensors for non-PreTrainedModel instances with tied weights,
        # even when save_safetensors=False is set in TrainingArguments.
        def custom_save_model(self, output_dir: Optional[str] = None, _internal_call: bool = False):
            _output_dir = Path(output_dir if output_dir is not None else self.args.output_dir)
            _output_dir.mkdir(parents=True, exist_ok=True)

            # Explicitly save model state dictionary using torch.save
            torch.save(self.model.state_dict(), _output_dir / "pytorch_model.bin")

            # Also save the tokenizer and trainer state if applicable
            # Use the tokenizer from the enclosing scope directly
            if tokenizer is not None: # Modified from self.tokenizer
                tokenizer.save_pretrained(_output_dir) # Modified from self.tokenizer
            if _internal_call: # Only save trainer_state.json for internal calls
                self.state.save_to_json(_output_dir / "trainer_state.json")

            # Log the save action
            print(f"Custom save_model: Model saved to {_output_dir / 'pytorch_model.bin'}")

        # Bind the custom save_model method to this specific trainer instance
        trainer.save_model = custom_save_model.__get__(trainer, Trainer)

        return trainer
    # --- End of local redefinitions for error fix ---

    # ------------------------------------------------------------------
    # 1. Ensure the input is a proper pandas DataFrame
    # ------------------------------------------------------------------
    if not isinstance(train_dev_df, pd.DataFrame):
        train_dev_df = pd.DataFrame(train_dev_df)

    train_dev_df = train_dev_df.reset_index(drop=True)

    # ------------------------------------------------------------------
    # 2. Basic validation
    # ------------------------------------------------------------------
    required_columns = {"label"}
    missing_columns = required_columns - set(train_dev_df.columns)

    if missing_columns:
        raise ValueError(
            f"train_dev_df is missing required column(s): {missing_columns}. "
            f"Available columns are: {list(train_dev_df.columns)}"
        )

    # Ensure labels are clean integers
    train_dev_df["label"] = train_dev_df["label"].astype(int)

    y = train_dev_df["label"].to_numpy()
    row_indices = np.arange(len(train_dev_df))

    print("Training/development dataframe shape:", train_dev_df.shape)
    print("Number of labels:", len(y))
    print("Label distribution:")
    print(train_dev_df["label"].value_counts().sort_index())

    # ------------------------------------------------------------------
    # 3. Check that each class has enough samples for stratified CV
    # ------------------------------------------------------------------
    min_class_count = train_dev_df["label"].value_counts().min()

    if min_class_count < N_SPLITS:
        raise ValueError(
            f"Cannot run StratifiedKFold with n_splits={N_SPLITS}. "
            f"The smallest class has only {min_class_count} samples."
        )

    # ------------------------------------------------------------------
    # 4. Run cross-validation
    # ------------------------------------------------------------------
    all_results = []

    for seed in CV_SEEDS:
        skf = StratifiedKFold(
            n_splits=N_SPLITS,
            shuffle=True,
            random_state=seed
        )

        for fold, (train_idx, val_idx) in enumerate(
            skf.split(row_indices, y),
            start=1
        ):
            print(f"\n===== Seed {seed} | Fold {fold}/{N_SPLITS} ====")

            fold_train = train_dev_df.iloc[train_idx].reset_index(drop=True)
            fold_val = train_dev_df.iloc[val_idx].reset_index(drop=True)

            print("Fold train shape:", fold_train.shape)
            print("Fold validation shape:", fold_val.shape)

            fold_dir = OUTPUT_DIR / f"cv_seed_{seed}_fold_{fold}"

            # Use the locally defined build_trainer_local
            trainer = build_trainer_local(
                train_df=fold_train,
                eval_df=fold_val,
                output_dir=fold_dir,
                epochs=EPOCHS_CV,
                seed=seed,
                eval_during_training=True,
            )

            trainer.train()
            metrics = trainer.evaluate()

            metrics.update({
                "seed": seed,
                "fold": fold
            })

            all_results.append(metrics)

            pd.DataFrame(all_results).to_csv(
                OUTPUT_DIR / "cv_metrics_running.csv",
                index=False
            )

    results_df = pd.DataFrame(all_results)
    results_df.to_csv(OUTPUT_DIR / "cv_metrics.csv", index=False)

    return results_df
print(type(train_dev_df),"*********")
print(train_dev_df.shape if hasattr(train_dev_df, "shape") else len(train_dev_df))
print(train_dev_df.head() if hasattr(train_dev_df, "head") else train_dev_df.keys())
# Uncomment to run CV.
cv_results = run_cross_validation(train_dev_df)
display(cv_results)
display(cv_results.describe(include="all"))


## 11. Final training on the full training/development split and one-time tag-free holdout evaluation

The final model is trained on the full training/development split without using the holdout set for epoch selection. The tag-free holdout is evaluated only after training has completed.

In [ ]:
def train_final_and_evaluate(train_dev_df: pd.DataFrame, holdout_df: pd.DataFrame, seed: int = 42):
    final_dir = OUTPUT_DIR / f"final_seed_{seed}"

    # Do not pass holdout_df as eval_df here. The holdout must not influence
    # checkpoint selection or early model choice.
    final_trainer = build_trainer(
        train_df=train_dev_df,
        eval_df=None,
        output_dir=final_dir,
        epochs=EPOCHS_FINAL,
        seed=seed,
        eval_during_training=False,
    )
    final_trainer.train()

    # One-time final evaluation on the separate tag-free holdout.
    holdout_dataset = SetswanaMRPDataset(holdout_df, tokenizer, max_length=MAX_LENGTH)
    holdout_metrics = trainer.evaluate(eval_dataset=holdout_dataset, metric_key_prefix="holdout")

    logits, probs, preds = predict_logits(trainer, holdout_df)
    y_true = holdout_df["label"].values

    print("\nHoldout metrics:")
    print(json.dumps(holdout_metrics, indent=2))
    print("\nClassification report:")
    print(classification_report(y_true, preds, target_names=["non_offensive", "offensive"], zero_division=0))
    print("\nConfusion matrix:")
    print(confusion_matrix(y_true, preds))

    #macro_ci = bootstrap_metric_ci(
     #   y_true,
      #  preds,
       # lambda a, b: f1_score(a, b, average="macro", zero_division=0),
        #n_resamples=BOOTSTRAP_RESAMPLES,
        #eed=seed,
    #)
    #mcc_ci = bootstrap_metric_ci(
     #   y_true,
      #  preds,
       # matthews_corrcoef,
        #n_resamples=BOOTSTRAP_RESAMPLES,
        #seed=seed,
    #)
    #print(f"Macro-F1 95% bootstrap CI: {macro_ci}")
    #print(f"MCC 95% bootstrap CI: {mcc_ci}")

    pred_frame = holdout_df.copy()
    pred_frame["pred"] = preds
    pred_frame["p_non_offensive"] = probs[:, 0]
    pred_frame["p_offensive"] = probs[:, 1]
    pred_frame.to_csv(OUTPUT_DIR / "holdout_predictions.csv", index=False)

    return trainer, pred_frame, holdout_metrics

# Uncomment for final run.
final_trainer, holdout_predictions, holdout_metrics = train_final_and_evaluate(train_dev_df, holdout_df, seed=42)


## 12. Counterfactual rationale deletion

This checks whether the model's offensive probability drops when the gold rationale span is removed. A meaningful drop suggests sensitivity to the annotated evidence. An excessively high flip rate, however, should be interpreted together with classification quality and rationale-alignment metrics.

In [ ]:
from pathlib import Path
from typing import List, Tuple, Optional
import re
import pandas as pd
import numpy as np


# ---------------------------------------------------------------------
# Optional holdout rationale metadata file
# ---------------------------------------------------------------------
# This file should correspond row-by-row to TEST_DATA_PATH, but with
# <TRIGGER>...</TRIGGER> markup preserved for offensive examples.
#
# The model will NOT be evaluated on this file directly.
# It is used only to recover gold rationale spans for counterfactual tests.
# ---------------------------------------------------------------------

#TEST_WITH_TRIGGERS_PATH = None
# Example:
TEST_WITH_TRIGGERS_PATH = "/content/drive/My Drive/Colab Notebooks/TEST_WITH_TRIGGERS.csv"


def read_table(path: str | Path) -> pd.DataFrame:
    """Read CSV, TSV, Excel, JSON, or JSONL into a dataframe."""
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"File does not exist: {path}")

    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)

    if suffix == ".tsv":
        return pd.read_csv(path, sep="\t")

    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)

    if suffix == ".jsonl":
        return pd.read_json(path, lines=True)

    if suffix == ".json":
        return pd.read_json(path)

    raise ValueError(f"Unsupported file type: {suffix}")


def normalise_for_matching(text: str) -> str:
    """Normalise text for sanity-check comparison between tag-free and cleaned trigger files."""
    text = "" if pd.isna(text) else str(text)
    return re.sub(r"\s+", " ", text).strip()


def detect_text_column(df: pd.DataFrame) -> str:
    """
    Detect likely text column.

    This keeps the cell flexible across slightly different dataset versions.
    """
    candidates = [
        "text",
        "TEXT",
        "sentence",
        "Sentence",
        "clean_text",
        "raw_text",
        "content",
        "tweet",
    ]

    for col in candidates:
        if col in df.columns:
            return col

    raise ValueError(
        "Could not detect text column in test_with_triggers file. "
        f"Available columns are: {list(df.columns)}"
    )


def detect_label_column(df: pd.DataFrame) -> Optional[str]:
    """Detect likely label column if present."""
    candidates = ["label", "LABEL", "class", "target", "y"]

    for col in candidates:
        if col in df.columns:
            return col

    return None


def extract_clean_text_and_spans_from_trigger_markup(
    marked_text: str,
) -> Tuple[str, List[Tuple[int, int]]]:
    """
    Convert trigger-marked text into clean text and character spans.

    Example:
        "Ke <TRIGGER>setlaela</TRIGGER> tota"

    becomes:
        clean_text = "Ke setlaela tota"
        char_spans = [(3, 11)]

    The returned spans refer to positions in the cleaned text, not the
    original trigger-marked text.
    """

    if pd.isna(marked_text):
        return "", []

    marked_text = str(marked_text)

    pattern = re.compile(
        r"<\s*TRIGGER\s*>(.*?)<\s*/\s*TRIGGER\s*>",
        flags=re.IGNORECASE | re.DOTALL,
    )

    clean_parts = []
    spans = []
    cursor = 0
    clean_cursor = 0

    for match in pattern.finditer(marked_text):
        # Add text before the trigger.
        before = marked_text[cursor:match.start()]
        clean_parts.append(before)
        clean_cursor += len(before)

        # Add trigger content, but without tags.
        trigger_text = match.group(1)
        span_start = clean_cursor
        clean_parts.append(trigger_text)
        clean_cursor += len(trigger_text)
        span_end = clean_cursor

        spans.append((span_start, span_end))

        cursor = match.end()

    # Add remaining text after the last trigger.
    after = marked_text[cursor:]
    clean_parts.append(after)

    clean_text = "".join(clean_parts)

    # Light whitespace normalisation changes character positions.
    # Therefore, we must rebuild spans under normalised whitespace.
    # To avoid span corruption, we do NOT aggressively normalise before
    # span creation above. However, downstream deletion is more robust if
    # text is clean. So we do a careful re-alignment below.
    clean_text_aligned, aligned_spans = realign_spans_after_whitespace_normalisation(
        clean_text,
        spans,
    )

    return clean_text_aligned, aligned_spans


def realign_spans_after_whitespace_normalisation(
    text: str,
    spans: List[Tuple[int, int]],
) -> Tuple[str, List[Tuple[int, int]]]:
    """
    Normalise whitespace while preserving span alignment.

    This builds a character-index mapping from original clean text to
    whitespace-normalised clean text.
    """

    if not text:
        return "", []

    new_chars = []
    old_to_new = {}
    new_idx = 0
    prev_space = False

    for old_idx, ch in enumerate(text):
        if ch.isspace():
            if not prev_space:
                new_chars.append(" ")
                old_to_new[old_idx] = new_idx
                new_idx += 1
                prev_space = True
            else:
                # Repeated whitespace maps to the current new index.
                old_to_new[old_idx] = new_idx
        else:
            new_chars.append(ch)
            old_to_new[old_idx] = new_idx
            new_idx += 1
            prev_space = False

    normalised = "".join(new_chars).strip()

    # Account for possible leading-space strip.
    leading_spaces = len("".join(new_chars)) - len("".join(new_chars).lstrip())
    if leading_spaces > 0:
        normalised_shift = leading_spaces
    else:
        normalised_shift = 0

    aligned_spans = []

    for start, end in spans:
        if start >= len(text) or end <= start:
            continue

        # Map start to first available character position.
        mapped_start = None
        for pos in range(start, min(end, len(text))):
            if pos in old_to_new:
                mapped_start = old_to_new[pos] - normalised_shift
                break

        # Map end to one past the last available character position.
        mapped_end = None
        for pos in range(min(end - 1, len(text) - 1), start - 1, -1):
            if pos in old_to_new:
                mapped_end = old_to_new[pos] + 1 - normalised_shift
                break

        if mapped_start is not None and mapped_end is not None:
            mapped_start = max(0, mapped_start)
            mapped_end = min(len(normalised), mapped_end)

            if mapped_end > mapped_start:
                aligned_spans.append((mapped_start, mapped_end))

    return normalised, aligned_spans


def load_holdout_rationale_metadata(
    test_with_triggers_path: str | Path,
    holdout_predictions: pd.DataFrame,
    id_col: Optional[str] = None,
) -> pd.DataFrame:
    """
    Load the trigger-marked holdout file and attach gold rationale spans
    to the existing tag-free holdout prediction frame.

    The trigger-marked file is used only for metadata extraction.
    It is not passed directly into the classifier.
    """

    trigger_df = read_table(test_with_triggers_path).copy()

    text_col = detect_text_column(trigger_df)
    label_col = detect_label_column(trigger_df)

    trigger_df["clean_text_from_triggers"], trigger_df["char_spans_from_triggers"] = zip(
        *trigger_df[text_col].map(extract_clean_text_and_spans_from_trigger_markup)
    )

    # ------------------------------------------------------------------
    # Case 1: merge by ID column if available
    # ------------------------------------------------------------------
    if id_col is not None and id_col in holdout_predictions.columns and id_col in trigger_df.columns:
        metadata = trigger_df[
            [id_col, "clean_text_from_triggers", "char_spans_from_triggers"]
            + ([label_col] if label_col is not None else [])
        ].copy()

        merged = holdout_predictions.merge(metadata, on=id_col, how="left")

    # ------------------------------------------------------------------
    # Case 2: assume row-by-row correspondence
    # ------------------------------------------------------------------
    else:
        if len(trigger_df) != len(holdout_predictions):
            raise ValueError(
                "The trigger-marked holdout file and holdout_predictions have different lengths. "
                f"test_with_triggers rows={len(trigger_df)}, "
                f"holdout_predictions rows={len(holdout_predictions)}. "
                "Provide a shared id_col or ensure both files are row-aligned."
            )

        merged = holdout_predictions.copy().reset_index(drop=True)
        merged["clean_text_from_triggers"] = trigger_df["clean_text_from_triggers"].values
        merged["char_spans_from_triggers"] = trigger_df["char_spans_from_triggers"].values

        if label_col is not None:
            merged["label_from_triggers_file"] = trigger_df[label_col].values

    # ------------------------------------------------------------------
    # Sanity check: cleaned trigger text should match tag-free text
    # ------------------------------------------------------------------
    if "clean_text" not in merged.columns:
        raise ValueError(
            "holdout_predictions must contain a 'clean_text' column. "
            f"Available columns are: {list(merged.columns)}"
        )

    left = merged["clean_text"].map(normalise_for_matching)
    right = merged["clean_text_from_triggers"].map(normalise_for_matching)

    mismatch_mask = left != right
    mismatch_count = int(mismatch_mask.sum())

    if mismatch_count > 0:
        print(
            f"WARNING: {mismatch_count}/{len(merged)} rows do not perfectly match "
            "after removing trigger tags."
        )
        print("Showing first 5 mismatches for inspection:")

        display(
            merged.loc[
                mismatch_mask,
                ["clean_text", "clean_text_from_triggers"]
            ].head()
        )

        # This is intentionally a warning rather than immediate failure,
        # because minor punctuation/spacing differences may exist.
        # If mismatches are substantial, stop and fix the file alignment.

    # Optional label sanity check if labels are available in both files.
    if "label_from_triggers_file" in merged.columns:
        label_mismatch = (
            merged["label"].astype(int).to_numpy()
            != merged["label_from_triggers_file"].astype(int).to_numpy()
        )

        if label_mismatch.any():
            raise ValueError(
                f"Label mismatch between tag-free holdout and trigger-marked holdout "
                f"in {int(label_mismatch.sum())} rows. Check file alignment."
            )

    # Attach spans under the name expected by the deletion function.
    merged["char_spans"] = merged["char_spans_from_triggers"]

    n_with_spans = merged["char_spans"].map(len).gt(0).sum()
    print(f"Attached rationale spans to holdout predictions.")
    print(f"Rows with at least one rationale span: {n_with_spans}/{len(merged)}")

    return merged


def delete_char_spans(text: str, spans: List[Tuple[int, int]]) -> str:
    """Remove character spans from text, preserving the remaining context."""

    text = "" if pd.isna(text) else str(text)

    if not spans:
        return text

    spans = sorted(spans, key=lambda x: x[0])

    parts = []
    cursor = 0

    for start, end in spans:
        start = max(0, int(start))
        end = min(len(text), int(end))

        if start < cursor:
            continue

        if end <= start:
            continue

        parts.append(text[cursor:start])
        cursor = end

    parts.append(text[cursor:])

    return re.sub(r"\s+", " ", "".join(parts)).strip()


def run_counterfactual_deletion(
    trainer: Trainer,
    holdout_pred_frame: pd.DataFrame,
    test_with_triggers_path: Optional[str | Path] = None,
    id_col: Optional[str] = None,
) -> pd.DataFrame:
    """
    Run counterfactual rationale deletion on correctly classified
    offensive examples.

    If test_with_triggers_path is supplied, rationale spans are first
    extracted from the corresponding trigger-marked holdout file and
    attached to holdout_pred_frame.

    The trainer remains the same final_trainer used for tag-free
    evaluation. No new trainer is required.
    """

    if test_with_triggers_path is not None:
        holdout_pred_frame = load_holdout_rationale_metadata(
            test_with_triggers_path=test_with_triggers_path,
            holdout_predictions=holdout_pred_frame,
            id_col=id_col,
        )

    if "char_spans" not in holdout_pred_frame.columns:
        raise ValueError(
            "No 'char_spans' column found. Provide test_with_triggers_path "
            "or attach rationale spans to holdout_predictions before calling this function."
        )

    subset = holdout_pred_frame[
        (holdout_pred_frame["label"] == 1)
        & (holdout_pred_frame["pred"] == 1)
        & (holdout_pred_frame["char_spans"].map(len) > 0)
    ].copy().reset_index(drop=True)

    print(f"Correctly classified offensive examples with rationale spans: {len(subset)}")

    if subset.empty:
        print("No correctly classified offensive examples with rationale spans found.")
        return subset

    counter_df = subset.copy()

    counter_df["clean_text_original"] = counter_df["clean_text"]

    counter_df["clean_text"] = [
        delete_char_spans(text, spans)
        for text, spans in zip(counter_df["clean_text"], counter_df["char_spans"])
    ]

    # Important: after deletion, there are no rationale spans left
    # for the counterfactual input.
    counter_df["char_spans"] = [[] for _ in range(len(counter_df))]

    _, counter_probs, counter_preds = predict_logits(trainer, counter_df)

    subset["p_offensive_after_deletion"] = counter_probs[:, 1]
    subset["pred_after_deletion"] = counter_preds
    subset["delta_p_offensive"] = (
        subset["p_offensive"] - subset["p_offensive_after_deletion"]
    )
    subset["label_flipped"] = subset["pred_after_deletion"] != subset["pred"]

    print("Mean Δp_offensive:", subset["delta_p_offensive"].mean())
    print("Median Δp_offensive:", subset["delta_p_offensive"].median())
    print("Label flip rate:", subset["label_flipped"].mean())

    subset.to_csv(
        OUTPUT_DIR / "counterfactual_rationale_deletion.csv",
        index=False,
    )

    return subset


# ---------------------------------------------------------------------
# Run counterfactual rationale deletion
# ---------------------------------------------------------------------
# Use the same final_trainer already used for tag-free holdout prediction.
# Supply the corresponding trigger-marked holdout file only as metadata.
# ---------------------------------------------------------------------

counterfactual_results = run_counterfactual_deletion(
    trainer=final_trainer,
    holdout_pred_frame=holdout_predictions,
    test_with_triggers_path=TEST_WITH_TRIGGERS_PATH,
    id_col=None,  # Set this if both files share a stable ID column.
)

display(counterfactual_results.head())


## 13. Optional LIME/S-LIME rationale-alignment scaffold

This section is intentionally separated from training. It reuses the companion repository's explanation philosophy: explain the final tag-free classifier and compare top-attributed tokens with the gold trigger-derived rationale set.

For a strict paper implementation, keep the exact same LIME/S-LIME settings used in the companion explainability notebooks.

In [ ]:
def make_predict_proba_fn(trainer: Trainer):
    """Return a function compatible with LIME: list[str] -> np.ndarray probabilities."""
    def predict_proba(texts: List[str]) -> np.ndarray:
        temp_df = pd.DataFrame({
            "clean_text": texts,
            "label": [0] * len(texts),
            "char_spans": [[] for _ in texts],
        })
        _, probs, _ = predict_logits(trainer, temp_df)
        return probs
    return predict_proba


def token_overlap_scores(gold_tokens: set, predicted_tokens: set) -> Dict[str, float]:
    """Compute rationale precision/recall/F1 over token sets."""
    if len(gold_tokens) == 0 or len(predicted_tokens) == 0:
        return {"rat_precision": 0.0, "rat_recall": 0.0, "rat_f1": 0.0}
    overlap = len(gold_tokens & predicted_tokens)
    precision = overlap / len(predicted_tokens)
    recall = overlap / len(gold_tokens)
    f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
    return {"rat_precision": precision, "rat_recall": recall, "rat_f1": f1}


def simple_gold_rationale_words(text: str, spans: List[Tuple[int, int]]) -> set:
    """Approximate word-level gold rationale set for LIME alignment."""
    words = []
    for start, end in spans:
        words.extend(re.findall(r"\w+", text[start:end].lower()))
    return set(words)


def run_lime_alignment(trainer: Trainer, holdout_pred_frame: pd.DataFrame, num_features: int = 10, num_samples: int = 1000):
    try:
        from lime.lime_text import LimeTextExplainer
    except ImportError as exc:
        raise ImportError("Install lime first: pip install lime") from exc

    explainer = LimeTextExplainer(class_names=["non_offensive", "offensive"])
    predict_proba = make_predict_proba_fn(trainer)

    subset = holdout_pred_frame[
        (holdout_pred_frame["label"] == 1)
        & (holdout_pred_frame["pred"] == 1)
        & (holdout_pred_frame["char_spans"].map(len) > 0)
    ].copy().reset_index(drop=True)

    rows = []
    for idx, row in subset.iterrows():
        text = row["clean_text"]
        gold_words = simple_gold_rationale_words(text, row["char_spans"])
        k = max(1, len(gold_words))

        exp = explainer.explain_instance(
            text,
            predict_proba,
            labels=[1],
            num_features=max(num_features, k),
            num_samples=num_samples,
        )
        top_words = [w.lower() for w, weight in exp.as_list(label=1)[:k]]
        pred_words = set(re.findall(r"\w+", " ".join(top_words).lower()))
        scores = token_overlap_scores(gold_words, pred_words)
        scores.update({
            "row_id": idx,
            "gold_words": sorted(gold_words),
            "pred_words": sorted(pred_words),
        })
        rows.append(scores)

    out = pd.DataFrame(rows)
    out.to_csv(OUTPUT_DIR / "lime_rationale_alignment.csv", index=False)
    if not out.empty:
        print(out[["rat_precision", "rat_recall", "rat_f1"]].mean())
    return out

# Uncomment after final evaluation.
lime_alignment = run_lime_alignment(final_trainer, holdout_predictions, num_features=10, num_samples=1000)
display(lime_alignment.head())

## 14. Save experiment card

This creates a small JSON artefact documenting exactly what was inherited and what was added in the MRP extension.

In [ ]:
def save_experiment_card() -> None:
    card = {
        "experiment": "PuoBERTa + Masked Rationale Prediction for Setswana offensive language detection",
        "inherits_from": "setswana-offensive-977 companion repository workflow",
        "model_name": MODEL_NAME,
        "max_length": MAX_LENGTH,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "epochs_cv": EPOCHS_CV,
        "epochs_final": EPOCHS_FINAL,
        "batch_train": BATCH_TRAIN,
        "batch_eval": BATCH_EVAL,
        "class_weights": CLASS_WEIGHTS,
        "cv_seeds": CV_SEEDS,
        "slime_seeds": SLIME_SEEDS,
        "source_protocol": "separate trigger-annotated train/development split and separate tag-free holdout/test split",
        "n_splits": N_SPLITS,
        "lambda_mrp": LAMBDA_MRP,
        "new_contribution": [
            "trigger spans converted to gold rationale supervision",
            "rationale-masked auxiliary MRP branch",
            "joint classification + MRP loss",
            "tag-free final holdout evaluation",
            "counterfactual rationale-deletion analysis",
            "optional LIME/S-LIME rationale-alignment analysis",
        ],
    }
    path = OUTPUT_DIR / "experiment_card_mrp_puoberta.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(card, f, indent=2)
    print(f"Saved: {path}")

# save_experiment_card()

## 15. Recommended execution order

1. Set `TRAIN_DATA_PATH` and `TEST_DATA_PATH`, or set `HF_DATASET_ID` with `HF_TRAIN_SPLIT` and `HF_TEST_SPLIT`.
2. Run imports and configuration.
3. Load the released training/development split with triggers and the separate tag-free test split.
4. Prepare both dataframes separately using `prepare_dataframe()`.
5. Verify that offensive training/development rows contain rationale spans.
6. Verify that the test/holdout file contains no visible `<TRIGGER>` markup.
7. Run 5-fold stratified CV on the training/development split.
8. Train the final model on the full training/development split.
9. Evaluate once on the separate tag-free holdout/test split.
10. Run counterfactual rationale deletion only if hidden rationale metadata is available for the holdout examples.
11. Run LIME/S-LIME rationale-alignment analysis using the same settings as the companion explainability notebooks.

The key methodological rule is: **trigger markup is used only to build MRP supervision during training; it is never exposed to the final classifier at holdout inference time**.


## LIME rationale visualisation for correctly classified offensive holdout samples

This section loops through correctly classified offensive holdout examples, uses LIME to extract **positive offensive-class contributors only**, compares them with the gold trigger-derived rationales, and saves an HTML report. The trigger-marked test file is used only as rationale metadata; the trained `final_trainer` remains the same tag-free MRP classifier.


In [ ]:
import html
import json
import re
import subprocess
import sys
from pathlib import Path
from typing import List, Tuple, Optional, Dict, Any

import numpy as np
import pandas as pd
import torch
from IPython.display import HTML, display

try:
    from lime.lime_text import LimeTextExplainer
except ImportError:
    print("Installing lime because it is not available in this runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "lime", "-q"])
    from lime.lime_text import LimeTextExplainer


# -----------------------------
# User-adjustable parameters
# -----------------------------
LIME_NUM_SAMPLES = 1000
LIME_NUM_FEATURES = 15
DISPLAY_LIMIT = 10  # display first N visualisations in notebook; full report is saved to HTML
CLASS_NAMES = ["non_offensive", "offensive"]
OFFENSIVE_LABEL = 1

# TEST_WITH_TRIGGERS_PATH should already be set in the notebook.
# Example:
# TEST_WITH_TRIGGERS_PATH = "/content/drive/MyDrive/.../test_with_triggers.csv"


# -----------------------------
# Basic helpers
# -----------------------------
def _normalise_token(token: str) -> str:
    """Normalise a LIME/gold token for word-level overlap comparison."""
    token = "" if token is None else str(token)
    token = token.strip().lower()
    # Keep word characters and hyphen/apostrophe-like internal characters lightly.
    token = re.sub(r"^[\w]+|[\w]+$", "", token, flags=re.UNICODE)
    return token


def _word_tokens(text: str) -> List[str]:
    """Simple word-level tokenisation for rationale-alignment summaries."""
    text = "" if pd.isna(text) else str(text)
    return [_normalise_token(t) for t in re.findall(r"\b\w+\b", text, flags=re.UNICODE) if _normalise_token(t)]


def tokens_from_char_spans(text: str, spans: List[Tuple[int, int]]) -> List[str]:
    """Extract normalised word tokens from gold character spans."""
    text = "" if pd.isna(text) else str(text)
    out = []
    for start, end in spans or []:
        start = max(0, int(start))
        end = min(len(text), int(end))
        if end > start:
            out.extend(_word_tokens(text[start:end]))
    return out


def safe_spans(spans: Any) -> List[Tuple[int, int]]:
    """Convert stored span values into a list of integer tuples."""
    if spans is None or (isinstance(spans, float) and np.isnan(spans)):
        return []
    if isinstance(spans, list):
        return [tuple(map(int, s)) for s in spans if len(s) == 2]
    if isinstance(spans, tuple):
        return [tuple(map(int, spans))] if len(spans) == 2 else []
    if isinstance(spans, str):
        try:
            parsed = json.loads(spans)
            return safe_spans(parsed)
        except Exception:
            try:
                import ast
                parsed = ast.literal_eval(spans)
                return safe_spans(parsed)
            except Exception:
                return []
    return []


def highlight_char_spans_html(text: str, spans: List[Tuple[int, int]]) -> str:
    """Return HTML text with gold rationale character spans highlighted."""
    text = "" if pd.isna(text) else str(text)
    spans = sorted(safe_spans(spans), key=lambda x: x[0])
    parts = []
    cursor = 0
    for start, end in spans:
        start = max(0, min(len(text), int(start)))
        end = max(0, min(len(text), int(end)))
        if end <= start or start < cursor:
            continue
        parts.append(html.escape(text[cursor:start]))
        parts.append(
            '<mark style="background:#ffe08a; padding:2px 4px; border-radius:4px;">'
            + html.escape(text[start:end])
            + '</mark>'
        )
        cursor = end
    parts.append(html.escape(text[cursor:]))
    return "".join(parts)


def highlight_lime_tokens_html(text: str, tokens: List[str]) -> str:
    """Highlight top positive offensive-class LIME tokens in text."""
    text = "" if pd.isna(text) else str(text)
    clean_tokens = sorted({t for t in [_normalise_token(t) for t in tokens] if t}, key=len, reverse=True)
    if not clean_tokens:
        return html.escape(text)

    pattern = re.compile(r"(?<!\w)(" + "|".join(re.escape(t) for t in clean_tokens) + r")(?![\w])", flags=re.IGNORECASE | re.UNICODE)
    parts = []
    cursor = 0
    for match in pattern.finditer(text):
        parts.append(html.escape(text[cursor:match.start()]))
        parts.append(
            '<mark style="background:#b8e6ff; padding:2px 4px; border-radius:4px;">'
            + html.escape(text[match.start():match.end()])
            + '</mark>'
        )
        cursor = match.end()
    parts.append(html.escape(text[cursor:]))
    return "".join(parts)


def _extract_logits(model_outputs: Any) -> torch.Tensor:
    """Extract classification logits from common Hugging Face/custom output structures."""
    if hasattr(model_outputs, "logits"):
        return model_outputs.logits
    if isinstance(model_outputs, dict):
        if "logits" in model_outputs:
            return model_outputs["logits"]
        if "cls_logits" in model_outputs:
            return model_outputs["cls_logits"]
    if isinstance(model_outputs, (tuple, list)):
        # Prefer a tensor shaped [batch, 2].
        for item in model_outputs:
            if torch.is_tensor(item) and item.ndim == 2 and item.shape[-1] == 2:
                return item
        # Otherwise assume first element after loss if present.
        if len(model_outputs) > 1 and torch.is_tensor(model_outputs[1]):
            return model_outputs[1]
        if len(model_outputs) > 0 and torch.is_tensor(model_outputs[0]):
            return model_outputs[0]
    raise ValueError(f"Could not extract logits from model output type: {type(model_outputs)}")


def predict_proba_texts(texts: List[str], batch_size: int = 32) -> np.ndarray:
    """
    Probability function used by LIME.
    It uses final_trainer.model directly on tag-free text.
    """
    model = final_trainer.model
    model.eval()
    device = next(model.parameters()).device

    all_probs = []
    for start in range(0, len(texts), batch_size):
        batch_texts = ["" if pd.isna(t) else str(t) for t in texts[start:start + batch_size]]
        encoded = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            # Corrected: Pass inputs as input_ids_cls and attention_mask_cls
            outputs = model(
                input_ids_cls=encoded["input_ids"],
                attention_mask_cls=encoded["attention_mask"],
            )
            logits = _extract_logits(outputs)
            probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
        all_probs.append(probs)

    return np.vstack(all_probs)


def ensure_holdout_with_rationales(holdout_predictions: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure holdout_predictions contains char_spans.
    If missing, attach them from TEST_WITH_TRIGGERS_PATH using the metadata loader defined earlier.
    """
    hp = holdout_predictions.copy()

    if "char_spans" in hp.columns and hp["char_spans"].map(lambda x: len(safe_spans(x))).sum() > 0:
        hp["char_spans"] = hp["char_spans"].map(safe_spans)
        return hp

    if "TEST_WITH_TRIGGERS_PATH" not in globals() or TEST_WITH_TRIGGERS_PATH in [None, ""]:
        raise ValueError(
            "holdout_predictions has no usable char_spans and TEST_WITH_TRIGGERS_PATH is not set. "
            "Set TEST_WITH_TRIGGERS_PATH to the corresponding test_with_triggers.csv file."
        )

    if "load_holdout_rationale_metadata" not in globals():
        raise ValueError(
            "load_holdout_rationale_metadata() is not defined. Run the counterfactual-deletion metadata cell first, "
            "or paste its helper functions before this visualisation cell."
        )

    hp = load_holdout_rationale_metadata(
        test_with_triggers_path=TEST_WITH_TRIGGERS_PATH,
        holdout_predictions=hp,
        id_col=None,
    )
    hp["char_spans"] = hp["char_spans"].map(safe_spans)
    return hp


def compute_alignment(gold_tokens: List[str], pred_tokens: List[str]) -> Dict[str, float]:
    """Word-level set overlap alignment metrics."""
    G = {t for t in gold_tokens if t}
    P = {t for t in pred_tokens if t}
    overlap = G.intersection(P)
    precision = len(overlap) / len(P) if len(P) else 0.0
    recall = len(overlap) / len(G) if len(G) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return {
        "rationale_precision": precision,
        "rationale_recall": recall,
        "rationale_f1": f1,
        "overlap_count": len(overlap),
    }


# -----------------------------
# Main LIME loop
# -----------------------------

def run_lime_rationale_visualisation_for_correct_offensive(
    holdout_predictions: pd.DataFrame,
    output_dir: Path = OUTPUT_DIR,
) -> pd.DataFrame:
    """
    Loop through all correctly classified offensive holdout samples with rationales,
    compute LIME explanations for class 1, and save visualisations.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    hp = ensure_holdout_with_rationales(holdout_predictions)

    required = {"clean_text", "label", "pred", "char_spans"}
    missing = required - set(hp.columns)
    if missing:
        raise ValueError(f"holdout_predictions is missing required columns: {missing}. Available: {list(hp.columns)}")

    subset = hp[
        (hp["label"].astype(int) == OFFENSIVE_LABEL)
        & (hp["pred"].astype(int) == OFFENSIVE_LABEL)
        & (hp["char_spans"].map(lambda x: len(safe_spans(x)) > 0))
    ].copy().reset_index(drop=True)

    print(f"Correctly classified offensive examples with gold rationale spans: {len(subset)}")
    if subset.empty:
        return subset

    explainer = LimeTextExplainer(
        class_names=CLASS_NAMES,
        random_state=42,
    )

    summary_rows = []
    html_blocks = []

    for i, row in subset.iterrows():
        text = str(row["clean_text"])
        spans = safe_spans(row["char_spans"])
        gold_tokens = tokens_from_char_spans(text, spans)
        k = max(1, len(gold_tokens))

        exp = explainer.explain_instance(
            text_instance=text,
            classifier_fn=predict_proba_texts,
            labels=[OFFENSIVE_LABEL],
            num_features=LIME_NUM_FEATURES,
            num_samples=LIME_NUM_SAMPLES,
        )

        lime_items = [(str(tok), float(weight)) for tok, weight in exp.as_list(label=OFFENSIVE_LABEL)]

        # IMPORTANT: rationale-alignment uses positive offensive-class contributors only.
        positive_items = [(tok, weight) for tok, weight in lime_items if weight > 0]
        positive_items = sorted(positive_items, key=lambda x: x[1], reverse=True)

        top_k_items = positive_items[:k]
        top_k_tokens = [_normalise_token(tok) for tok, _ in top_k_items if _normalise_token(tok)]

        align = compute_alignment(gold_tokens, top_k_tokens)

        p_off = row.get("p_offensive", np.nan)
        summary = {
            "index_in_subset": i,
            "original_holdout_index": row.get("index", row.get("row_id", np.nan)),
            "clean_text": text,
            "p_offensive": p_off,
            "gold_tokens": gold_tokens,
            "top_positive_offensive_lime_tokens": top_k_tokens,
            "top_positive_offensive_lime_items": top_k_items,
            "all_lime_items_class_1": lime_items,
            "k": k,
            **align,
        }
        summary_rows.append(summary)

        gold_html = highlight_char_spans_html(text, spans)
        lime_html = highlight_lime_tokens_html(text, top_k_tokens)
        weights_html = "".join(
            f"<tr><td>{html.escape(str(tok))}</td><td>{weight:.6f}</td></tr>"
            for tok, weight in top_k_items
        ) or '<tr><td colspan="2"><em>No positive offensive-class LIME tokens selected.</em></td></tr>'

        block = f"""
        <div style="border:1px solid #ddd; border-radius:10px; padding:14px; margin:14px 0; font-family:Arial, sans-serif;">
            <h3 style="margin-top:0;">Example {i+1}</h3>
            <p><strong>Model offensive probability:</strong> {p_off if not pd.isna(p_off) else 'N/A'}</p>
            <p><strong>Gold rationale span(s):</strong><br>{gold_html}</p>
            <p><strong>Top positive offensive-class LIME token(s), k = |G|:</strong><br>{lime_html}</p>
            <p><strong>Gold tokens:</strong> {html.escape(str(gold_tokens))}</p>
            <p><strong>Selected LIME tokens:</strong> {html.escape(str(top_k_tokens))}</p>
            <p><strong>Alignment:</strong> P={align['rationale_precision']:.3f}, R={align['rationale_recall']:.3f}, F1={align['rationale_f1']:.3f}</p>
            <table style="border-collapse:collapse; margin-top:8px;">
                <thead>
                    <tr><th style="border:1px solid #ccc; padding:4px 8px;">LIME token</th><th style="border:1px solid #ccc; padding:4px 8px;">Positive class-1 weight</th></tr>
                </thead>
                <tbody>{weights_html}</tbody>
            </table>
        </div>
        """
        html_blocks.append(block)

        if (i + 1) % 10 == 0:
            print(f"Processed {i+1}/{len(subset)} examples...")

    summary_df = pd.DataFrame(summary_rows)

    # Save CSV summary. Lists are stringified by pandas, which is acceptable for inspection.
    csv_path = output_dir / "mrp_lime_rationale_alignment_summary.csv"
    summary_df.to_csv(csv_path, index=False)

    # Save full HTML report.
    report_html = f"""
    <html>
    <head>
        <meta charset="utf-8">
        <title>MRP PuoBERTa LIME Rationale Visualisation</title>
    </head>
    <body style="font-family:Arial, sans-serif; max-width:1100px; margin:30px auto; line-height:1.5;">
        <h1>MRP PuoBERTa: LIME Rationale Visualisation</h1>
        <p>This report includes correctly classified offensive holdout examples with gold rationale spans.</p>
        <p><strong>Important:</strong> predicted rationale tokens are the top positive LIME contributors for class 1 (offensive) only. Negative class-1 weights are excluded from the predicted rationale set.</p>
        <p><strong>Gold rationale highlight:</strong> <mark style="background:#ffe08a; padding:2px 4px; border-radius:4px;">yellow</mark></p>
        <p><strong>LIME selected-token highlight:</strong> <mark style="background:#b8e6ff; padding:2px 4px; border-radius:4px;">blue</mark></p>
        <hr>
        {''.join(html_blocks)}
    </body>
    </html>
    """

    html_path = output_dir / "mrp_lime_rationale_visualisations.html"
    html_path.write_text(report_html, encoding="utf-8")

    print(f"Saved summary CSV to: {csv_path}")
    print(f"Saved full HTML report to: {html_path}")

    # Display aggregate alignment summary.
    print("\nAggregate word-level rationale alignment against top positive offensive-class LIME tokens:")
    display(summary_df[["rationale_precision", "rationale_recall", "rationale_f1", "overlap_count", "k"]].describe())

    # Display first few visualisations only to keep notebook manageable.
    display(HTML("".join(html_blocks[:DISPLAY_LIMIT])))

    return summary_df


# Run the visualisation loop.
# Ensure final_trainer, holdout_predictions, tokenizer, MAX_LENGTH, and TEST_WITH_TRIGGERS_PATH are defined.
mrp_lime_rationale_summary = run_lime_rationale_visualisation_for_correct_offensive(
    holdout_predictions=holdout_predictions,
    output_dir=OUTPUT_DIR,
)

display(mrp_lime_rationale_summary.head())